# Imports

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

from uavloc.features.segmentation import CLASSES, load_unet
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.plotting.segmentation import colour_labels, fixed_broken
from uavloc.segmentation.metrics import iou_scores
from uavloc.track import REPO_ROOT

sys.path.insert(0, str(REPO_ROOT / "pocs" / "village-seg-baseline"))
import village_seg as vs

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None

# Params

In [ ]:
TAG = "round2"
HOLDOUT_VILLAGE = "bushcha-50p2422-25p8305"
DRONE_VAL_BLOCKS = [0, 4]
DRONE_GSD_M = 0.10
DRONE_EXAMPLES = 4
SAT_WINDOW_PX = 1024

# Data

In [ ]:
DERIVED = REPO_ROOT / "data" / "derived"
HERE = REPO_ROOT / "pocs" / "village-seg-baseline"
FIGS = HERE / "figures"
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
DEVICE

In [ ]:
sat = vs.load_satellite(DERIVED / "ua-village-annotated", DERIVED)
drone = vs.load_drone(DERIVED / "ua-drone-annotated", DERIVED)
vs.split_by_footprint(drone, DRONE_VAL_BLOCKS)
drone_val = [d for d in drone if d.split == "val"]

In [ ]:
SETS = {
    "bushcha": {"domain": "satellite", "items": [(s.img, s.lab) for s in sat if s.loc == HOLDOUT_VILLAGE],
                "names": [s.date for s in sat if s.loc == HOLDOUT_VILLAGE]},
    "drone-val": {"domain": "drone", "items": [vs.drone_at_gsd(d, DRONE_GSD_M) for d in drone_val],
                  "names": [d.name[-8:] for d in drone_val]},
}
{n: len(s["items"]) for n, s in SETS.items()}

# Model

In [ ]:
MODELS = {
    "shared BN": load_unet(HERE / f"model-{TAG}.pt", DEVICE),
    "satellite BN": load_unet(HERE / f"model-{TAG}-bn-satellite.pt", DEVICE),
    "drone BN": load_unet(HERE / f"model-{TAG}-bn-drone.pt", DEVICE),
}

# Run

In [ ]:
preds = {name: {m: [vs.predict(model, img, DEVICE) for img, _ in s["items"]] for m, model in MODELS.items()}
         for name, s in SETS.items()}

In [ ]:
pd.DataFrame({m: {name: iou_scores(preds[name][m], [lab for _, lab in s["items"]])["miou"]
                  for name, s in SETS.items()} for m in MODELS}).round(3)

In [ ]:
per_image = pd.DataFrame([
    {"set": name, "image": s["names"][i],
     **{m: iou_scores([preds[name][m][i]], [lab])["miou"] for m in MODELS}}
    for name, s in SETS.items() for i, (_, lab) in enumerate(s["items"])
]).set_index(["set", "image"])
per_image.assign(gain=per_image[["satellite BN", "drone BN"]].max(axis=1) - per_image["shared BN"]).round(3)

# Plots

In [ ]:
def centre_window(arr: np.ndarray, size_px: int) -> np.ndarray:
    top, left = max((arr.shape[0] - size_px) // 2, 0), max((arr.shape[1] - size_px) // 2, 0)
    return arr[top:top + size_px, left:left + size_px]

In [ ]:
def plot_bn_examples(name: str, rows: list[int], window_px: int | None) -> plt.Figure:
    """Image, label, shared BN, own-domain BN, what own BN fixed (green) / broke (red), other-domain BN."""
    own = f"{SETS[name]['domain']} BN"
    other = "drone BN" if own == "satellite BN" else "satellite BN"
    crop = (lambda a: centre_window(a, window_px)) if window_px else (lambda a: a)
    cols = ["image", "label", "shared BN", own, "own BN vs shared: fixed (green) / broke (red)", other]
    fig, axes = plt.subplots(len(rows), len(cols), figsize=(3.4 * len(cols), 3.0 * len(rows)), squeeze=False)
    for r, i in enumerate(rows):
        img, lab = SETS[name]["items"][i]
        p = {m: preds[name][m][i] for m in MODELS}
        panels = [img, colour_labels(lab), colour_labels(p["shared BN"]), colour_labels(p[own]),
                  fixed_broken(lab, p["shared BN"], p[own]), colour_labels(p[other])]
        for ax, col, panel in zip(axes[r], cols, panels):
            ax.imshow(crop(panel)); ax.set_xticks([]); ax.set_yticks([])
            score = per_image.loc[(name, SETS[name]["names"][i]), col] if col in MODELS else None
            ax.set_title(col if score is None else f"{col}  mIoU {score:.2f}", fontsize=9)
        axes[r, 0].set_ylabel(SETS[name]["names"][i], fontsize=9)
    legend = [plt.Rectangle((0, 0), 1, 1, color=SEG_PALETTE[c] / 255) for c in range(len(CLASSES))]
    fig.legend(legend, CLASSES, loc="lower center", ncol=len(CLASSES))
    fig.suptitle(f"{name}: same weights, BatchNorm statistics from different domains")
    fig.tight_layout(rect=(0, .05, 1, 1))
    return fig

In [ ]:
drone_rows = np.linspace(0, len(SETS["drone-val"]["items"]) - 1, DRONE_EXAMPLES).astype(int).tolist()
fig = plot_bn_examples("drone-val", drone_rows, None)
fig.savefig(FIGS / f"{TAG}-bn-example-drone.png", dpi=90, bbox_inches="tight")

In [ ]:
fig = plot_bn_examples("bushcha", list(range(len(SETS["bushcha"]["items"]))), SAT_WINDOW_PX)
fig.savefig(FIGS / f"{TAG}-bn-example-bushcha.png", dpi=90, bbox_inches="tight")